# Yahtzee state explorer

Run this notebook from the root of your Yahtzee project, next to files like
`value_iteration.py`, `precomputed.py`, `constants.py`, and the `data/` folder.

It assumes you have already run value iteration and have files like:

`data/values/level_kk/<13-bit mask>.pkl`

The helpers below let you inspect a reduced game state, a roll, the optimal
first keep, second keep, final category choice, and nearby alternatives.

In [ ]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd

from constants import *
from precomputed import (
    ALL_DICE_STATES, ALL_DICE_FREQS, ALL_KEEPS, KEEP_IDX, KEEPS_FOR_DICE,
    REROLL_OUTCOMES, DICE_IDX,
    dice_values_to_idx, dice_idx_to_values, dice_idx_to_vec, dice_vec_to_idx,
    SCORE_ROWS, JOKER_SCORE_ROWS, IS_YAHTZEE_T, YAHTZEE_FACE_T,
)
from reduced_game_state import ReducedGameState
from value_iteration import mask_path, VALUES_DIR, REROLL_MATRIX, REROLL_OFFSETS, REROLL_PAIR_KEEPS

## Basic display helpers

In [ ]:
def cat_name(c):
    return CATEGORY_NAMES[int(c)]

def mask_from_categories(categories):
    '''
    categories can be ints or names from CATEGORY_NAMES.
    Example:
        mask_from_categories(["Ones", "Twos", YAHTZEE])
    '''
    mask = 0
    for c in categories:
        if isinstance(c, str):
            c = CATEGORY_NAMES.index(c)
        mask |= 1 << int(c)
    return mask

def categories_from_mask(mask):
    return [CATEGORY_NAMES[c] for c in range(NUM_CATEGORIES) if mask & (1 << c)]

def keep_to_values(keep_idx):
    keep = ALL_KEEPS[int(keep_idx)]
    return tuple(face for face, count in enumerate(keep, start=1) for _ in range(int(count)))

def vec_to_values(vec):
    return tuple(face for face, count in enumerate(vec, start=1) for _ in range(int(count)))

def roll_label(dice_idx):
    return dice_idx_to_values(int(dice_idx))

def state_level(state):
    return int(state.filled_mask).bit_count()

def describe_state(state):
    return {
        "filled_mask": f"{state.filled_mask:013b}",
        "level": state_level(state),
        "filled": categories_from_mask(state.filled_mask),
        "upper_total": state.upper_total,
        "yahtzee_eligible": bool(state.yahtzee_eligible),
    }

## Load the value-iteration payload for one state

Each payload file contains all reduced states with the same `filled_mask`.
Rows are ordered by `(upper_total, yahtzee_eligible)`, matching how
`process_mask` sorted the states before saving.

In [ ]:
def load_payload_for_state(state):
    level = state_level(state)
    path = Path(mask_path(level, state.filled_mask))
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Run value_iteration for level {level}, "
            "or check that this notebook is running from the project root."
        )
    with path.open("rb") as f:
        payload = pickle.load(f)
    return payload

def row_index_for_state(payload, state):
    key = (int(state.upper_total), bool(state.yahtzee_eligible))
    indices = [(int(u), bool(e)) for u, e in payload["indices"]]
    try:
        return indices.index(key)
    except ValueError:
        raise KeyError(f"State row {key} not found. Available rows include: {indices[:10]} ...")

def get_state_row(state):
    payload = load_payload_for_state(state)
    row = row_index_for_state(payload, state)
    return payload, row

def state_value(state):
    payload, row = get_state_row(state)
    return float(payload["V"][row])

## Inspect decisions for a specific state and roll

In [ ]:
def inspect_roll(state, roll):
    '''
    roll can be a raw dice tuple/list like [1, 1, 3, 5, 6],
    or a dice_idx.
    '''
    dice_idx = int(roll) if isinstance(roll, (int, np.integer)) else dice_values_to_idx(roll)
    payload, row = get_state_row(state)

    keep_A = int(payload["decisions_A"][row, dice_idx])
    keep_B = int(payload["decisions_B"][row, dice_idx])
    cat_C = int(payload["decisions_C"][row, dice_idx])

    return pd.DataFrame([
        {
            "stage": "A: before first reroll",
            "roll": roll_label(dice_idx),
            "best_action": f"keep {keep_to_values(keep_A)}",
            "action_raw": ALL_KEEPS[keep_A],
            "EV": float(payload["ev_A"][row, dice_idx]),
        },
        {
            "stage": "B: before second reroll",
            "roll": roll_label(dice_idx),
            "best_action": f"keep {keep_to_values(keep_B)}",
            "action_raw": ALL_KEEPS[keep_B],
            "EV": float(payload["ev_B"][row, dice_idx]),
        },
        {
            "stage": "C: choose category",
            "roll": roll_label(dice_idx),
            "best_action": cat_name(cat_C),
            "action_raw": cat_C,
            "EV": float(payload["ev_C"][row, dice_idx]),
        },
    ])

## Rank first-keep or second-keep alternatives for a roll

This recomputes the EV of each legal keep from the stored downstream EV table.
For stage A, the downstream table is `ev_B`; for stage B, it is `ev_C`.

In [ ]:
def keep_alternatives(state, roll, stage="A"):
    dice_idx = int(roll) if isinstance(roll, (int, np.integer)) else dice_values_to_idx(roll)
    payload, row = get_state_row(state)

    if stage.upper() == "A":
        downstream = payload["ev_B"][row]
    elif stage.upper() == "B":
        downstream = payload["ev_C"][row]
    else:
        raise ValueError("stage must be 'A' or 'B'")

    rows = []
    for keep_idx in KEEPS_FOR_DICE[dice_idx]:
        finals, nums = REROLL_OUTCOMES[(dice_idx, int(keep_idx))]
        ev = sum(downstream[fi] * n for fi, n in zip(finals, nums)) / 7776.0
        rows.append({
            "keep_idx": int(keep_idx),
            "keep": keep_to_values(keep_idx),
            "keep_vec": ALL_KEEPS[int(keep_idx)],
            "EV": float(ev),
        })

    df = pd.DataFrame(rows).sort_values("EV", ascending=False).reset_index(drop=True)
    df["EV_gap_from_best"] = df["EV"].iloc[0] - df["EV"]
    return df

## Rank final category alternatives for a roll

This mirrors the logic in `_stage_C`, but for a single state and roll,
so you can see why the category choice was made.

In [ ]:
def legal_categories_for_state_and_roll(state, dice_idx):
    return state.legal_categories_by_idx(dice_idx)

def category_alternatives(state, roll):
    dice_idx = int(roll) if isinstance(roll, (int, np.integer)) else dice_values_to_idx(roll)
    payload, row = get_state_row(state)

    # Load next-level value arrays by category.
    # Missing next states only happen at terminal, where continuation value is zero.
    next_level = state_level(state) + 1
    V_next_by_mask = {}
    next_dir = Path(VALUES_DIR) / f"level_{next_level:02d}"
    if next_dir.exists():
        for fn in next_dir.glob("*.pkl"):
            mask = int(fn.stem, 2)
            with fn.open("rb") as f:
                p = pickle.load(f)
            arr = np.zeros((UPPER_BONUS_THRESHOLD + 1, 2), dtype=np.float32)
            for (u, e), v in zip(p["indices"], p["V"]):
                arr[int(u), int(e)] = v
            V_next_by_mask[mask] = arr

    is_joker, categories = legal_categories_for_state_and_roll(state, dice_idx)
    score_row = JOKER_SCORE_ROWS[dice_idx] if is_joker else SCORE_ROWS[dice_idx]

    rows = []
    for c in categories:
        points = int(score_row[c])
        reward = points

        if c <= SIXES:
            new_upper = min(state.upper_total + points, UPPER_BONUS_THRESHOLD)
            if state.upper_total < UPPER_BONUS_THRESHOLD and state.upper_total + points >= UPPER_BONUS_THRESHOLD:
                reward += UPPER_BONUS
        else:
            new_upper = state.upper_total

        if is_joker and state.yahtzee_eligible:
            reward += EXTRA_YAHTZEE_BONUS

        if c == YAHTZEE and points == YAHTZEE_POINTS:
            new_eligible = True
        else:
            new_eligible = bool(state.yahtzee_eligible)

        new_mask = state.filled_mask | (1 << c)
        continuation = 0.0
        if new_mask in V_next_by_mask:
            continuation = float(V_next_by_mask[new_mask][new_upper, int(new_eligible)])

        rows.append({
            "category": cat_name(c),
            "category_idx": c,
            "is_joker": bool(is_joker),
            "score_points": points,
            "immediate_reward": reward,
            "new_upper": new_upper,
            "new_eligible": new_eligible,
            "continuation_EV": continuation,
            "total_EV": reward + continuation,
        })

    df = pd.DataFrame(rows).sort_values("total_EV", ascending=False).reset_index(drop=True)
    df["EV_gap_from_best"] = df["total_EV"].iloc[0] - df["total_EV"]
    return df

## Find interesting rolls in a state

This lists rolls where the optimal first keep is close to another option,
which is often useful for debugging strategy behavior.

In [ ]:
def closest_first_keep_margins(state, n=20):
    rows = []
    for dice_idx in range(len(ALL_DICE_STATES)):
        alts = keep_alternatives(state, dice_idx, stage="A")
        if len(alts) < 2:
            continue
        rows.append({
            "roll": roll_label(dice_idx),
            "best_keep": alts.loc[0, "keep"],
            "best_EV": alts.loc[0, "EV"],
            "second_keep": alts.loc[1, "keep"],
            "second_EV": alts.loc[1, "EV"],
            "margin": alts.loc[0, "EV"] - alts.loc[1, "EV"],
        })
    return pd.DataFrame(rows).sort_values("margin").head(n).reset_index(drop=True)

## Example usage

Edit these examples to match the state you care about.

In [ ]:
# Example: after Ones and Twos have been filled, with upper_total=6,
# and no scored Yahtzee yet.
state = ReducedGameState(
    filled_mask=mask_from_categories(["Ones", "Twos"]),
    upper_total=6,
    yahtzee_eligible=False,
)

describe_state(state), state_value(state)

In [ ]:
inspect_roll(state, [1, 1, 3, 5, 6])

In [ ]:
keep_alternatives(state, [1, 1, 3, 5, 6], stage="A").head(15)

In [ ]:
keep_alternatives(state, [1, 1, 3, 5, 6], stage="B").head(15)

In [ ]:
category_alternatives(state, [1, 1, 3, 5, 6])

In [ ]:
closest_first_keep_margins(state, n=20)